# ***Building RAYA — The Architect of Type 1 Civilization***

 “Driving humanity toward a sustainable and intelligent civilization.”



---

### 🌍 ***Problem Statement: "The Human Sustainability Index (HSI) 2.0 — AI for a Livable Future"***

The next frontier for humanity isn’t just technological progress — it’s **sustainable survival**.

**HSI 2.0** acts as an intelligent sustainability compass — an AI-powered mirror that reflects how livable and future-ready a city truly is. It goes beyond static metrics to **analyze, predict, and simulate** the balance between human development and natural systems.

By integrating multi-dimensional data on **water, energy, climate, pollution, and waste**, HSI 2.0 identifies **emerging risks, regional disparities, and pathways for sustainable growth**.

Through **AI-driven clustering, predictive modeling, and generative insights**, it empowers **governments, communities, and organizations** to design smarter policies, foster resilience, and ensure a **thriving planet for generations ahead.** 🌱✨

---



----

### **What's in Human Sustainability Index (HSI) 2.0 ?**

**Ans:** This dataset offers a realistic and holistic foundation for Human Sustainability Index (HSI) prediction. It has been synthesized by integrating multiple open-source datasets to create a unified, systematized dataset that reflects real-world sustainability conditions across regions.

Instead of using a direct HSI score, we employed **clustering techniques** to determine optimal category groupings for HSI, resulting in five meaningful sustainability classes — **["Moderately Sustainable", "Critical (Unsustainable)", "Highly Sustainable", "Sustainable", "Low Sustainable"]**.

Additionally, we computed the **Urbanization %** and enriched the dataset with extended features such as **Energy Source, Water Consumption, SGD %, AQI, AQI Bucket, Waste Type, Disposal Method, Recycling Rate (%),** and **Cost of Waste Management (₹/Ton)** — all contributing to a more data-driven and actionable sustainability assessment.



In [51]:
import pandas as pd
import numpy as np

data = pd.read_csv('RAYA HSI.csv')

data.tail()

,Date,State,City,Population,EnergySource,WaterConsumption,SGD %,AQI,AQI_Bucket,Waste Type,Disposal Method,Recycling Rate (%),Cost of Waste Management(?/Ton)
20085,31-12-2024,Uttar Pradesh,Noida,712593,Hydro,146.54,20.86,111,Moderate,Organic,Landfill,52.60,2606.29
20086,31-12-2024,Uttar Pradesh,Ghaziabad,1740165,Solar,187.39,21.25,198,Moderate,Organic,Composting,46.46,4413.87
20087,31-12-2024,Uttar Pradesh,Ghazipur,141519,Wind,143.42,13.14,340,Very Poor,E-Waste,Incineration,33.94,2763.71
20088,31-12-2024,Uttar Pradesh,Lucknow,3214849,Wind,177.43,11.57,108,Moderate,Plastic,Landfill,29.81,3785.70
20089,31-12-2024,Uttar Pradesh,Meerut,1417862,Wind,100.29,18.78,86,Satisfactory,Plastic,Composting,15.67,2301.46


In [2]:
data.isnull().sum()

,0
Date,0
State,0
City,0
Population,0
EnergySource,0
WaterConsumption,0
SGD %,0
AQI,0
AQI_Bucket,0
Waste Type,0


In [52]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select features for clustering
features = data[['Population',
 'WaterConsumption', 'SGD %', 'AQI',
 'Recycling Rate (%)', 'Cost of Waste Management(?/Ton)']]


# -----------------------------
# Scale features — very important for K-Means
# -----------------------------
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [53]:
# -----------------------------
# Find optimal k with Elbow Method
# -----------------------------
inertia = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(features_scaled)
    inertia.append(kmeans.inertia_)  # Distortion / SSE

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=list(k_range),
        y=inertia,
        mode='lines+markers',
        marker=dict(color='royalblue', size=8),
        line=dict(width=2),
        name='Inertia'
    )
)

fig.update_layout(
    title='Elbow Method For Optimal k',
    xaxis_title='Number of clusters (k)',
    yaxis_title='Inertia (SSE)',
    xaxis=dict(tickmode='linear'),
    template='plotly_white',
    width=800,
    height=500
)

fig.show()

In [54]:
# Suppose you choose k= from the elbow
optimal_k = 5

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42)
data['HSI_Type'] = kmeans_final.fit_predict(features_scaled)

print(data.groupby('HSI_Type')[['Population', 'WaterConsumption', 'SGD %', 'AQI', 'Recycling Rate (%)', 'Cost of Waste Management(?/Ton)']].mean())

            Population  WaterConsumption      SGD %         AQI  \
HSI_Type                                                          
0         1.001271e+06        121.884530  14.659392  239.818715   
1         9.908597e+05        140.135015  15.092110  240.954000   
2         1.007336e+06        205.885331  14.788722  241.615686   
3         9.580294e+05        191.720617  15.206336  252.293574   
4         3.200275e+06        165.334372  15.102571  244.631606   

          Recycling Rate (%)  Cost of Waste Management(?/Ton)  
HSI_Type                                                       
0                  27.655551                      3620.926269  
1                  48.085197                      2184.774375  
2                  27.461226                      2482.162625  
3                  46.454814                      4052.992635  
4                  37.416759                      3096.931110  


In [55]:
cluster_avg = data.groupby("HSI_Type")[['Population', 'WaterConsumption', 'SGD %', 'AQI', 'Recycling Rate (%)', 'Cost of Waste Management(?/Ton)']].mean().sort_values(by='Population')
ordered_clusters = cluster_avg.index.tolist()

In [56]:
ordered_clusters

[3, 1, 0, 2, 4]

In [57]:
# Make a mapping
segment_names = ["Sustainable", "Low Sustainable", "Critical (Unsustainable)", "Moderately Sustainable", "Highly Sustainable"]
cluster_to_label = {cluster: segment_names[i] for i, cluster in enumerate(ordered_clusters)}

# Apply mapping
data["HSI_Label"] = data["HSI_Type"].map(cluster_to_label)

print(data[["City","HSI_Type", "HSI_Label"]].head())
print('--' * 25)
print(data[["HSI_Type", "HSI_Label"]].value_counts())

        City  HSI_Type           HSI_Label
0      Noida         1     Low Sustainable
1  Ghaziabad         1     Low Sustainable
2   Ghazipur         3         Sustainable
3    Lucknow         4  Highly Sustainable
4     Meerut         3         Sustainable
--------------------------------------------------
HSI_Type  HSI_Label               
3         Sustainable                 4217
0         Critical (Unsustainable)    4093
1         Low Sustainable             4000
4         Highly Sustainable          3955
2         Moderately Sustainable      3825
Name: count, dtype: int64


In [11]:
data.EnergySource.value_counts()

,count
EnergySource,
Hydro,4102
Mixed,4029
Wind,4007
Solar,3995
Coal,3957


In [58]:
data['EnergySource'] = data['EnergySource'].map({
    'Hydro': 1,
    'Mixed': 2,
    'Wind': 3,
    'Solar': 4,
    'Coal': 5
})

In [12]:
data['Waste Type'].value_counts()

,count
Waste Type,
Mixed,4066
Biomedical,4055
Plastic,4001
E-Waste,3995
Organic,3973


In [59]:
data['Waste Type'] = data['Waste Type'].map({
    'Mixed': 1,
    'Biomedical': 2,
    'Plastic': 3,
    'E-Waste': 4,
    'Organic': 5

})

In [13]:
data['Disposal Method'].value_counts()

,count
Disposal Method,
Incineration,5089
Composting,5069
Landfill,4979
Recycling,4953


In [60]:
data['Disposal Method'] = data['Disposal Method'].map({
    'Incineration': 1,
    'Composting': 2,
    'Landfill': 3,
    'Recycling': 4

})

In [24]:
data.head(8)

,Date,State,City,Population,EnergySource,WaterConsumption,SGD %,AQI,AQI_Bucket,Waste Type,Disposal Method,Recycling Rate (%),Cost of Waste Management(?/Ton),HSI_Type,HSI_Label
0,01-01-2014,Uttar Pradesh,Noida,682409,2,157.81,13.69,336,Very Poor,5,3,53.72,1714.49,1,Low Sustainable
1,01-01-2014,Uttar Pradesh,Ghaziabad,1666415,2,131.03,13.50,45,Good,5,4,29.81,1420.08,1,Low Sustainable
2,01-01-2014,Uttar Pradesh,Ghazipur,135421,5,196.98,17.18,98,Satisfactory,3,3,27.65,4655.29,3,Sustainable
3,01-01-2014,Uttar Pradesh,Lucknow,3157542,5,88.13,20.69,366,Very Poor,3,2,20.75,3593.62,4,Highly Sustainable
4,01-01-2014,Uttar Pradesh,Meerut,1392132,5,184.71,14.38,84,Satisfactory,4,2,43.27,4877.01,3,Sustainable
5,02-01-2014,Uttar Pradesh,Noida,746721,2,238.74,7.82,176,Moderate,1,4,28.71,3561.80,2,Moderately Sustainable
6,02-01-2014,Uttar Pradesh,Ghaziabad,1652356,3,97.11,5.85,302,Very Poor,1,1,52.23,4386.01,0,Critical (Unsustainable)
7,02-01-2014,Uttar Pradesh,Ghazipur,96485,2,139.82,12.60,271,Poor,1,2,52.35,2391.31,1,Low Sustainable


In [61]:
data_ts = data.copy()

In [ ]:
data.columns

Index(['Year', 'State', 'District', 'population', 'population_proper',
       'Location', 'EnergySource', 'WaterConsumption', 'SGD %', 'AQI',
       'AQI_Bucket', 'Waste Type', 'Disposal Method', 'Recycling Rate (%)',
       'Cost of Waste Management (₹/Ton)', 'Urbanization %', 'HSI_Type',
       'HSI_Label'],
      dtype='object')

In [26]:
df_numeric = data.select_dtypes(include=['number'])
corr = df_numeric.corr()

import plotly.express as px

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale='Teal',
    title="Correlation Heatmap (Numeric Columns Only)"
)

fig.update_layout(
    width=850,
    height=600
)

fig.show()


In [27]:
corr_features = [
    'Population',
       'EnergySource', 'WaterConsumption', 'SGD %', 'AQI',
       'Waste Type', 'Disposal Method', 'Recycling Rate (%)',
       'Cost of Waste Management(?/Ton)', 'HSI_Type'
]

import plotly.express as px
import plotly.graph_objects as go

# Calculate correlation matrix again
corr_matrix = data[corr_features].corr()

# Select only correlations with target column
target_corr = corr_matrix[['HSI_Type']].drop(index='HSI_Type').reset_index()
target_corr.columns = ['Feature', 'Correlation']
target_corr['AbsCorrelation'] = target_corr['Correlation'].abs()

# Sort by absolute correlation
target_corr = target_corr.sort_values(by='AbsCorrelation', ascending=False)

fig = px.bar(
    target_corr,
    x='Feature',
    y='Correlation',
    color='Correlation',
    title='Feature Correlations with Human Sustainability Index (HSI)',
    color_continuous_scale='Teal',
    text=target_corr['Correlation'].round(3)
)

fig.update_layout(
    width=1050,
    height=650,
    xaxis_title='Feature',
    yaxis_title='Correlation with Human Sustainability Index (HSI)',
    bargap=0.3
)

fig.update_traces(
    textposition='outside'
)

fig.show()


In [62]:
from sklearn.preprocessing import LabelEncoder

le_state = LabelEncoder()
le_city = LabelEncoder()

data['State_encoded'] = le_state.fit_transform(data['State'])
data['City_encoded'] = le_city.fit_transform(data['City'])


In [63]:
data.drop(['State', 'City'], axis=1, inplace=True)


In [64]:
from sklearn.model_selection import train_test_split

X = data[['State_encoded', 'City_encoded','Population',
       'EnergySource', 'WaterConsumption', 'SGD %', 'AQI',
       'Waste Type', 'Disposal Method', 'Recycling Rate (%)',
       'Cost of Waste Management(?/Ton)']]

y = data['HSI_Type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [65]:
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import plotly.figure_factory as ff

# ✅ Train LightGBM model (no SMOTE)
lgb_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,            # LightGBM handles depth automatically
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='multiclass',
    num_class=len(np.unique(y_train))
)

# ✅ Fit the model
lgb_model.fit(X_train, y_train)

# ✅ Predictions
y_pred = lgb_model.predict(X_test)

# ✅ Accuracy
print('----' * 16)
print(f"✅ LightGBM Classifier Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print('----' * 16)

# ✅ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

labels = ['Critical (Unsustainable)', 'Low Sustainable', 'Moderately Sustainable', 'Sustainable', 'Highly Sustainable']
z_text = [[str(y) for y in x] for x in cm]

# ✅ Plot confusion matrix using Plotly
fig = ff.create_annotated_heatmap(
    z=cm,
    x=labels,
    y=labels,
    annotation_text=z_text,
    colorscale='teal',
    showscale=True
)

fig.update_layout(
    title_text='Confusion Matrix - LightGBM Classifier',
    width=900,
    height=500
)

fig['data'][0]['showscale'] = True
fig.show()


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001904 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1541
[LightGBM] [Info] Number of data points in the train set: 16072, number of used features: 10
[LightGBM] [Info] Start training from score -1.593819
[LightGBM] [Info] Start training from score -1.617058
[LightGBM] [Info] Start training from score -1.657357
[LightGBM] [Info] Start training from score -1.551834
[LightGBM] [Info] Start training from score -1.630311
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
----------------------------------------------------------------
✅ LightGBM Classifier Accuracy: 0.98
----------------------------------------------------------------


In [ ]:
#from sklearn.metrics import classification_report

#print(classification_report(y_test,y_pred))


#### **Confusion matrix results (in context of Highly Sustainable) :-**

* **True Positive (TP) —**
  *“Highly Sustainable” correctly predicted as “Highly Sustainable”*
  → **806**

* **False Positive (FP) —**
  *Other classes incorrectly predicted as “Highly Sustainable”*
  (e.g., *Critical, Low Sustainable, Moderately Sustainable, Sustainable → Highly Sustainable*)
  → **1 + 1 + 4 + 1 = 7**

* **False Negative (FN) —**
  *“Highly Sustainable” incorrectly predicted as another class*
  (e.g., *Highly Sustainable → Low / Moderate / Sustainable / Critical*)
  → **1**

* **True Negative (TN) —**
  *All other class predictions correctly NOT labeled as “Highly Sustainable”*
  → This equals all remaining correct predictions across other classes:

  * Critical → Critical: **810**
  * Low Sustainable → Low Sustainable: **794**
  * Moderately Sustainable → Moderately Sustainable: **741**
  * Sustainable → Sustainable: **794**

---



In [66]:

# ✅ Get feature importance directly from LightGBM model
importance_df = pd.DataFrame({
    'Feature': lgb_model.feature_name_,
    'Importance': lgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

# ✅ Plot with Plotly in teal theme
fig = px.bar(
    importance_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title='💠 Feature Importance - LightGBM',
    color='Importance',
    color_continuous_scale='Tealgrn'
)

# ✅ Style
fig.update_layout(
    width=850,
    height=600,
    title_x=0.5,
    yaxis=dict(autorange="reversed"),
    xaxis_title="Importance Score",
    yaxis_title="Features",
    plot_bgcolor="white"
)

fig.show()


In [67]:
X_test['State_encoded'] = le_state.inverse_transform(X_test['State_encoded'])
X_test['City_encoded'] = le_city.inverse_transform(X_test['City_encoded'])


In [68]:
X_test['Predicted_HSI'] = y_pred

In [69]:
# HSI mapping
HSI_map = {
    2: 'Moderately Sustainable',
    0: 'Critical (Unsustainable)',
    4: 'Highly Sustainable',
    3: 'Sustainable',
    1: 'Low Sustainable'
}

Disposal_Method_map = {
    1: 'Incineration',
    2: 'Composting',
    3: 'Landfill',
    4: 'Recycling'

}

Waste_Type_map = {
    1: 'Mixed',
    2: 'Biomedical',
    3: 'Plastic',
    4: 'E-Waste',
    5: 'Organic'
}

Energy_Source_map = {
    1: 'Hydro',
    2: 'Mixed',
    3: 'Wind',
    4: 'Solar',
    5: 'Coal'

}



X_test['Predicted_HSI'] = X_test['Predicted_HSI'].map(HSI_map)
X_test['Disposal Method'] = X_test['Disposal Method'].map(Disposal_Method_map)
X_test['Waste Type'] = X_test['Waste Type'].map(Waste_Type_map)
X_test['EnergySource'] = X_test['EnergySource'].map(Energy_Source_map)


#### **Final Test Data** : Ready for semi-deployment

In [39]:
X_test.columns

Index(['State_encoded', 'City_encoded', 'Population', 'EnergySource',
       'WaterConsumption', 'SGD %', 'AQI', 'Waste Type', 'Disposal Method',
       'Recycling Rate (%)', 'Cost of Waste Management(?/Ton)',
       'Predicted_HSI'],
      dtype='object')

#### **Renaming Columns**

In [40]:
X_test.columns = [
    'State', 'City', 'Population',
    'Energy Source', 'Water Consumption', 'SGD %', 'AQI', 'Waste Type',
       'Disposal Method', 'Recycling Rate (%)',
       'Cost of Waste Management (₹/Ton)', 'Predicted HSI'
]

### *Semi-Deployment*

In [42]:
import pandas as pd
from ipywidgets import interact, widgets, VBox


# ------------------------------------------
# 🔹 Dropdown widgets
# ------------------------------------------
state_dropdown = widgets.Dropdown(
    options=sorted(X_test['State'].unique().tolist()),
    description='Select State:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

city_dropdown = widgets.Dropdown(
    options=[],
    description='Select City:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

output = widgets.Output()

# ------------------------------------------
# 🔁 Update districts dynamically
# ------------------------------------------
def update_cities(*args):
    selected_state = state_dropdown.value
    filtered_cities = X_test[X_test['State'] == selected_state]['City'].unique().tolist()
    city_dropdown.options = sorted(filtered_cities)
    if filtered_cities:
        city_dropdown.value = filtered_cities[0]  # Default to first city
        show_city_info(None)  # ✅ Show results immediately after state change

state_dropdown.observe(update_cities, 'value')

# ------------------------------------------
# 📊 Display selected city data
# ------------------------------------------
def show_city_info(change):
    with output:
        output.clear_output()
        selected_state = state_dropdown.value
        selected_city = city_dropdown.value

        row = X_test[(X_test['State'] == selected_state) & (X_test['City'] == selected_city)]
        if row.empty:
            print("No data found for this district.")
            return

        row = row.iloc[0]
        print(f"🏙️ City: {row['City']}, State: {row['State']}\n")

        print(f"👥 Population: {row['Population']:,}")
        print(f"⚡ Energy Source: {row['Energy Source']}")
        print(f"💧 Water Consumption: {row['Water Consumption']} liters/day (or unit in dataset)")
        print(f"🎯 SGD % Achievement: {row['SGD %']}%")
        print(f"🌫️ Air Quality Index (AQI): {row['AQI']}")
        print(f"🗑️ Waste Type: {row['Waste Type']}")
        print(f"🏗️ Disposal Method: {row['Disposal Method']}")
        print(f"♻️ Recycling Rate: {row['Recycling Rate (%)']}%")
        print(f"💰 Waste Management Cost: ₹{row['Cost of Waste Management (₹/Ton)']:,} per ton")

        print(f"🌍 Predicted HSI Category: {row['Predicted HSI']}")


city_dropdown.observe(show_city_info, 'value')

# ------------------------------------------
# 🚀 Initialize and display
# ------------------------------------------
update_cities()  # ✅ Run once to initialize first state + city data
display(VBox([state_dropdown, city_dropdown, output]))



---

### 🌍 **Future Work: Towards Intelligent Sustainability for Human Life**

At the final stage/upcoming stage, an **LLM-powered intelligence layer** will transform complex data into **human-readable insights and localized recommendations**, helping **states and cities** enhance livability, align human progress with nature, and take smarter steps toward a **truly sustainable civilization**. ✨



---

